### Inspect PC variation between sparse lms, dense corresp, NSM latents
---
Do spearman's rank heatmap and PC traversal grid plot. Need to run save_pc_snapshots.py and pc_snapshot_grid.py to build NSM traversal plot, then stitch it together with lm based warp plots below.

### 1. Setup and paths

In [ ]:
# Imports and paths
import os, re, json, ast
import torch
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from NSM.plotting import load_mrk_json
from NSM.morphometrics import *

# Specify training directory and atlas directory
RUN          = "run_v72"                      # training attempt directory
ATLAS_RUN    = "2026_07-15_13_06_22/"  # atlas/builder run that produced alignedLMs
DROPBOX_ROOT = Path("/home/k.wolcott/UFL Dropbox/Katherine Wolcott/neural_shape_models/final_dataset_aug26/atlas/")

# Build other directories relative to those above
cwd      = Path.cwd()
base_wd  = cwd.parent
train_dir = base_wd / RUN
os.chdir(train_dir)
print(f"Working directory: {os.getcwd()}")

LM_DIR      = DROPBOX_ROOT / ATLAS_RUN / "alignedLMs"
ATLAS_DIR   = DROPBOX_ROOT / ATLAS_RUN / "atlas"
MEAN_LMS_FN = ATLAS_DIR / "atlas_sparse_landmarks.mrk.json"
CKPT        = "2500"

OUT_DIR = Path("gmm_results_lms_v_lats")
OUT_DIR.mkdir(exist_ok=True)
print(f"Outputs will be written to: {OUT_DIR.resolve()}")

# Load config and parse species / vertebra from filenames (same logic as PCA_tSNE_UMAP.ipynb)
config_path = "model_params_config.json"
with open(config_path) as f:
    cfg = json.load(f)
print(f"\033[92mLoaded config from {config_path}\033[0m")

# Parse filenames
all_vtk_files = [os.path.basename(f) for f in cfg["list_mesh_paths"]]
print(f"{len(all_vtk_files)} meshes listed in config")

pat = re.compile(r"^(?P<species>[\w\s\-]+)[\-_ ]+[\w\d]+[\-_ ]+(?P<vertebra>[CTL]?\d+)", re.IGNORECASE)
labels, unmatched_files = [], []
for f in all_vtk_files:
    m = pat.match(os.path.basename(f))
    if m:
        labels.append((m.group("species").strip(), m.group("vertebra").strip()))
    else:
        labels.append((None, None))
        unmatched_files.append(f)

print(f"Parsed {sum(1 for s, v in labels if s)} / {len(labels)} filenames")
if unmatched_files:
    print(f"\033[33mUnmatched ({len(unmatched_files)}), first 5:\033[0m", unmatched_files[:5])

# Load NSM latent codes
CKPT_PATH = f"latent_codes/{CKPT}.pth"
latent_ckpt = torch.load(CKPT_PATH, map_location="cpu")
codes = latent_ckpt["latent_codes"]["weight"].detach().cpu().numpy()
print(f"Latent codes: {codes.shape}")
assert len(codes) == len(all_vtk_files), (
    f"{len(codes)} latent codes but {len(all_vtk_files)} meshes in config -- order/count mismatch")

In [ ]:
# Build specimen metadata table -- species, family, life history, vertebral region

SPECIES_CSV = "../lizard_species_list.csv"

# Parse specimen ID and vertebra from filenames
parsed = [pat.match(f) for f in all_vtk_files]
specimens = pd.DataFrame({"mesh":        all_vtk_files,
                          "specimen_id": [m.group("species") if m else None for m in parsed],
                          "vertebra":    [m.group("vertebra").upper() if m else None for m in parsed]})
print(f"Parsed {specimens['specimen_id'].notna().sum()} / {len(specimens)} filenames")

# Join against the species master list
sdf = pd.read_csv(SPECIES_CSV)
sdf["marker"] = sdf["marker"].astype(str).str.strip().str.strip("'\"")
sdf["color"]  = sdf["color"].apply(ast.literal_eval)
specimens = specimens.merge(sdf, left_on="specimen_id", right_on="specimen", how="left")

unmatched = specimens.loc[specimens["family"].isna(), "specimen_id"].dropna().unique()
print(f"{specimens['family'].notna().sum()} / {len(specimens)} specimens matched to {SPECIES_CSV}")
if len(unmatched):
    print(f"\033[33mUnmatched specimen IDs ({len(unmatched)}):\033[0m {sorted(unmatched)[:10]}")

# Region from vertebra letter code
REGION_NAMES = {"C": "CERVICAL", "T": "THORACIC", "L": "LUMBAR"}
specimens["region"] = specimens["vertebra"].str[0].map(REGION_NAMES)

print("Specimens dataframe head:\n", specimens.head())

# One color per trait, derived from that trait's marker
markers = ['P', '+', 's', 'd', 'X', 'o', '2']
colors = [(0.65, 0.69, 0.12),   # pea soup
        (0.84, 0.65, 0.23),   # saffron
        (0.72, 0.44, 0.22),   # mud
        (0.36, 0.557, 0.68),  # powder blue
        (0.10, 0.51, 0.40),   # deep blue
        (0.60, 0.50, 0.46),   # slate
        (0, 0, 0)]            # black
marker_to_color = dict(zip(markers, colors))

trait_marker = specimens.drop_duplicates("trait").set_index("trait")["marker"]
trait_colors = {t: marker_to_color.get(m, (0.5, 0.5, 0.5))
                for t, m in trait_marker.items() if pd.notna(t)}
specimens["trait_color"] = specimens["trait"].map(trait_colors)
unmapped = [t for t, m in trait_marker.items() if pd.notna(t) and m not in marker_to_color]
if unmapped:
    print(f"\033[33mTraits using a marker not in marker_to_color (defaulting to grey): {unmapped}\033[0m")
print(f"{len(trait_colors)} traits: {sorted(trait_colors)}")

### Load landmarks (`.mrk.json`) and build the shape array

In [ ]:
# Load 3D Slicer Atlas aligned and scaled landmark data
lm_coords = []
for fpath in all_vtk_files:
    lm_name = os.path.splitext(fpath)[0] + ".mrk.json"
    lm_path = LM_DIR / lm_name
    coords, _ = load_mrk_json(lm_path)
    lm_coords.append(coords)

lm_coords_3d = np.stack(lm_coords)   # (N, p, 3)
print(f"Landmark data shape - 3d: {lm_coords_3d.shape}")

# Atlas mean sparse landmarks (same landmark set as LM_DIR, from the setup cell)
mean_lms_3d, _ = load_mrk_json(MEAN_LMS_FN)
print(f"Atlas mean landmarks shape: {mean_lms_3d.shape}")
assert mean_lms_3d.shape == lm_coords_3d.shape[1:], (
    f"Atlas has {mean_lms_3d.shape[0]} landmarks but specimens have {lm_coords_3d.shape[1]} "
    f"-- wrong atlas run, or dense vs sparse mismatch")

In [ ]:
# Data checks before analysis

# Are the configurations already centred and scaled?
cs = np.array([centroid_size(X) for X in lm_coords_3d])
print(f"Centroid size: mean={cs.mean():.5f}  sd={cs.std():.5f}  CV={100*cs.std()/cs.mean():.2f}%  "
      f"range=({cs.min():.5f}, {cs.max():.5f})")
if 100 * cs.std() / cs.mean() > 5:
    print("\033[33mCentroid size varies by >5% — the configurations may not be fully scaled. "
          "Consider dividing each by its centroid size before proceeding.\033[0m")

consensus = mshape(lm_coords_3d)
print(f"\nConsensus vs atlas mean landmarks: Procrustes distance = "
      f"{procrustes_dist(consensus, mean_lms_3d):.6f}")
print(f"Mean per-landmark offset = {np.linalg.norm(consensus - mean_lms_3d, axis=1).mean():.6f}")

d_mean = dist_to_mean(lm_coords_3d)
print(f"\nProcrustes distance to consensus: mean={d_mean.mean():.5f}  "
      f"median={np.median(d_mean):.5f}  max={d_mean.max():.5f}")

# Optional: rescale to unit centroid size (set to True if the check above complained)
RESCALE_TO_UNIT_CS = False
if RESCALE_TO_UNIT_CS:
    lm_coords_3d = np.stack([(X - X.mean(axis=0)) / centroid_size(X) for X in lm_coords_3d])
    consensus = mshape(lm_coords_3d)
    print("\nRescaled all configurations to unit centroid size.")

In [ ]:
# Identify which coordinate index is x (the axis of bilateral symmetry), y, and z
# Same checks as glassboxUMAP.ipynb, condensed. Adjust the index lists for your landmark scheme.

midline_idx = [20, 21, 8, 9, 12, 15]                                   # TO DO: midline landmarks (0-based)
paired_landmarks = {0: 5, 1: 4, 2: 3, 18: 19, 22: 23, 24: 25,
                    6: 7, 26: 27, 13: 14, 10: 11, 16: 17}              # TO DO: left: right pairs (0-based)
COTYLE_IDX, NEURAL_SPINE_IDX = 15, 12                                  # TO DO: for the y-axis check
ZYG_PR_IDX, ZYG_PO_IDX = 16, 0                                         # TO DO: for the z-axis check

mid = lm_coords_3d[:, midline_idx, :]   # (N, M, 3)
mid_flat = mid.reshape(-1, 3)
axis_var = np.var(mid_flat, axis=0)

# 1st check - Find axis of symmetry using landmarks along midline as constant
print("1st Check - Find which idx is the X-axis")
print(f"Midline Landmarks {midline_idx}")
x_idx = np.argmin(axis_var)
print("idx with the lowest var should be X axis (midline): X idx = ", x_idx)
print("0:", axis_var[0])
print("1:", axis_var[1])
print("2:", axis_var[2])

# 2nd check - Find which axis of symmetry varies most among paired landmarks (should be x; same as above)
print("\n2nd Check - Find which idx is the X-axis")
diffs = []
for left, right in paired_landmarks.items():
    L = lm_coords_3d[:, left, :]   # (N, 3)
    R = lm_coords_3d[:, right, :]  # (N, 3)
    d = L - R                      # (N, 3)
    diffs.append(d)
diffs = np.concatenate(diffs, axis=0)  # (N_pairs * N, 3)
axis_asym = np.var(diffs, axis=0)

print(f"Paired left-right landmarks {paired_landmarks}")
print("idx with the highest var should be X axis (landmarks are mirrored across X axis): X idx = ", np.argmax(axis_asym))
print("0:", axis_asym[0])
print("1:", axis_asym[1])
print("2:", axis_asym[2])

# 3rd check - Find which axis varies most among LM 13 - 16 (condyle and neural spine; should be y)
print("\n3rd Check - Find which idx is the Y-axis")
print(f"\nCotyle vs Neural Spine landmarks (13 & 16)")
L = lm_coords_3d[:,  NEURAL_SPINE_IDX, :]   
R = lm_coords_3d[:, COTYLE_IDX, :]
diff = L - R   
mean_abs = np.mean(np.abs(diff), axis=0)
y_idx = np.argmax(mean_abs)
print("idx with the highest var should be Y axis (LM): Y idx = ", y_idx)
print("0:", mean_abs[0])
print("1:", mean_abs[1])
print("2:", mean_abs[2])

# 4th check - Find which axis varies most among LM 1 - 17 (pre and post zygapophyses; should be z)
print("\n4th Check - Find which idx is the Z-axis")
print(f"\nPre- and post-zygapophyses landmarks (1 & 17)")
L = lm_coords_3d[:, ZYG_PO_IDX, :]   
R = lm_coords_3d[:, ZYG_PR_IDX, :]
diff = L - R   
mean_abs = np.mean(np.abs(diff), axis=0)
z_idx = np.argmax(mean_abs)
print("idx with the highest var should be Z axis (LM): Z idx = ", z_idx)
print("0:", mean_abs[0])
print("1:", mean_abs[1])
print("2:", mean_abs[2])

# Build axis list using determined order
axis_order = [None, None, None]
axis_order[z_idx] = 'z'
axis_order[x_idx] = 'x'
axis_order[y_idx] = 'y'

In [ ]:
# Symmetric vs asymmetric
mirrored = lm_coords_3d.copy()
mirrored[:, :, x_idx] *= -1
for l, r in paired_landmarks.items():
    mirrored[:, [l, r], :] = mirrored[:, [r, l], :]

symm_component = (lm_coords_3d + mirrored) / 2.0
asym_component = (lm_coords_3d - mirrored) / 2.0
da_component   = asym_component.mean(axis=0, keepdims=True)     # directional asymmetry
fa_component   = asym_component - da_component                  # fluctuating asymmetry

# Variance ratios relative to symmetric component
sym_var = np.var(symm_component)
da_var  = np.var(da_component)
fa_var  = np.var(fa_component)

da_ratio = da_var / sym_var
fa_ratio = fa_var / sym_var

print(f"DA / symmetric variance ratio: {da_ratio:.4f}")
print(f"FA / symmetric variance ratio: {fa_ratio:.4f}")

# Decision threshold — analogous to checking p-values in geomorph ANOVA
THRESHOLD = 0.05  # i.e., asymmetry explains < 5% of individual shape variation

if fa_ratio < THRESHOLD and da_ratio < THRESHOLD:
    print("Asymmetry signal is negligible — proceed with symmetric component only")
    coords_analysis = symm_component
else:
    print("Asymmetry signal is meaningful — retain both components")
    coords_analysis = lm_coords_3d 

### Check morphological disparity

In [ ]:
# ── Disparity: latents vs landmarks, with dimensionality robustness check ─────
# Procrustes variance is summed over dimensions, so the 512-dim latent space
# and 84-dim landmark configuration aren't comparable on raw values. Ranks
# within each space are the comparable unit — this cell confirms the family
# rank order is stable when latent dimensionality is matched to landmarks.

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.decomposition import PCA

ITER    = 999
N_MATCH = 84          # landmark dimensionality (28 LMs × 3 coords)
MIN_N   = 5           # disparity is unreliable below this

# ── Representations ───────────────────────────────────────────────────────────
L_std = (codes - codes.mean(0)) / (codes.std(0) + 1e-12)
Y_lm  = two_d_array(coords_analysis)

# PCA-reduced latents at matched dimensionality
pca_L   = PCA(n_components=N_MATCH).fit(L_std)
L_84    = pca_L.transform(L_std)
var_exp = pca_L.explained_variance_ratio_.sum()

print(f"Landmarks:      {Y_lm.shape}")
print(f"Latents (full): {L_std.shape}")
print(f"Latents ({N_MATCH} PCs): {L_84.shape}  ({100*var_exp:.1f}% of latent variance)")

REPS = [
    ("Landmarks",            Y_lm),
    ("Latents_full",         L_std),
    (f"Latents_{N_MATCH}PC", L_84),
]

# ── Disparity per group in each representation ───────────────────────────────
for col in ["family", "trait"]:
    keep = specimens[col].notna().values

    # Drop groups with too few specimens
    counts = specimens.loc[keep, col].value_counts()
    small  = counts[counts < MIN_N].index.tolist()
    if small:
        print(f"  Dropping n < {MIN_N}: {[(g, int(counts[g])) for g in small]}")
        keep &= ~specimens[col].isin(small).values

    groups_sub = specimens.loc[keep, col].values

    comp = pd.DataFrame()
    for rep_name, Y in REPS:
        var_tab, _, _ = morphol_disparity(Y[keep], groups_sub, iter=ITER)
        comp[f"var_{rep_name}"]  = var_tab
        comp[f"rank_{rep_name}"] = var_tab.rank(ascending=False).astype(int)

    # Drop singleton groups — zero variance is an artifact, not a finding
    singletons = comp[comp.filter(like="var_").sum(axis=1) == 0].index.tolist()
    if singletons:
        print(f"\n  Dropping singleton groups (zero variance): {singletons}")
        comp = comp.drop(index=singletons)
        for rep_name, _ in REPS:
            comp[f"rank_{rep_name}"] = comp[f"var_{rep_name}"].rank(ascending=False).astype(int)

    comp["shift_vs_LM"] = comp["rank_Landmarks"] - comp[f"rank_Latents_{N_MATCH}PC"]
    comp = comp.sort_values("rank_Landmarks")

    # Key check: do full and reduced latents agree on the ordering?
    rho_dim, p_dim = spearmanr(comp["rank_Latents_full"],
                               comp[f"rank_Latents_{N_MATCH}PC"])
    rho_lm,  p_lm  = spearmanr(comp["rank_Landmarks"],
                               comp[f"rank_Latents_{N_MATCH}PC"])

    print(f"\n{'='*70}")
    print(f"  Disparity by {col}  (n groups = {len(comp)})")
    print(f"{'='*70}")
    print(f"  Full vs {N_MATCH}-PC latents:  Spearman r = {rho_dim:+.3f}, p = {p_dim:.4f}")
    print(f"    → r > 0.9 means the rank order is robust to dimensionality")
    print(f"  Landmarks vs {N_MATCH}-PC latents: Spearman r = {rho_lm:+.3f}, p = {p_lm:.4f}")

    show = ["rank_Landmarks", "rank_Latents_full",
            f"rank_Latents_{N_MATCH}PC", "shift_vs_LM"]
    print(f"\n{comp[show].to_string()}")

    # Families the two representations disagree about most
    big = comp[comp["shift_vs_LM"].abs() >= 8].sort_values("shift_vs_LM", ascending=False)
    if len(big):
        print(f"\n  Largest disagreements (|shift| ≥ 8):")
        for grp, r in big.iterrows():
            direction = "MORE" if r["shift_vs_LM"] > 0 else "less"
            print(f"    {grp:<22s} {direction:>4s} disparate in latent space  "
                  f"(LM {int(r['rank_Landmarks']):>2d} → "
                  f"latent {int(r[f'rank_Latents_{N_MATCH}PC']):>2d})")

    comp.to_csv(OUT_DIR / f"disparity_dimcheck_{col}.csv")

print(f"\nSaved → {OUT_DIR.resolve()}")

In [ ]:
# ── Dumbbell plots: disparity rank shift, landmarks vs NSM latents ────────────
# One row per group; dots mark rank in each representation, the connecting line
# shows the shift. Rank-based, so the differing variance scales don't matter.
#
# Colours follow the same logic as the LDA figures: trait_colors, derived from
# each trait's marker in specimens. Families are coloured by their modal trait,
# so the ecological pattern reads directly off the figure.
from matplotlib.lines import Line2D

N_MATCH     = 84
RANK_COL    = f"rank_Latents_{N_MATCH}PC"
MIN_SHIFT   = 8       # groups below this are muted and unlabelled
GREY        = (0.62, 0.62, 0.62)
PURE_THRESH = 0.80    # families above this are drawn as one solid colour

# ── Family → trait composition, for striped colouring ────────────────────────
# Families are rarely single-strategy. Rather than collapsing to the mode (which
# would paint a 53/46 Scincidae as purely burrowing), mixed families are striped.
family_comp = (
    specimens.dropna(subset=["family", "trait"])
             .groupby("family")["trait"]
             .value_counts(normalize=True)
             .unstack(fill_value=0)
)
# Column order follows trait_colors so segments stack consistently
family_comp = family_comp[[t for t in trait_colors if t in family_comp.columns]]

family_trait = family_comp.idxmax(axis=1).to_dict()    # dominant trait, for sorting


def _segments(name, level, min_frac=0.05):
    """
    [(fraction, colour), ...] summing to 1, ordered by trait_colors.
    Mixed families split evenly between the traits they contain — the stripe
    flags that the family is heterogeneous, it doesn't encode the proportions.
    """
    if level == "trait":
        return [(1.0, trait_colors.get(name, GREY))]
    if name not in family_comp.index:
        return [(1.0, GREY)]

    comp = family_comp.loc[name]
    if comp.max() >= PURE_THRESH:
        return [(1.0, trait_colors.get(comp.idxmax(), GREY))]

    cols = [trait_colors.get(t, GREY) for t, f in comp.items() if f >= min_frac]
    return [(1.0 / len(cols), c) for c in cols]


def _group_color(name, level):
    """Dominant colour — used for dots and text, where striping isn't possible."""
    return _segments(name, level)[0][1] if level == "trait" \
        else trait_colors.get(family_trait.get(name), GREY)


def dumbbell(csv_path, out_path, level, min_shift=None, figsize=(9, 11)):
    """
    level: "family" or "trait" — controls how each row is coloured.
    min_shift: groups below this |shift| are drawn grey and unlabelled.
    """
    df = pd.read_csv(csv_path, index_col=0)

    if level == "family":
        # Group families by dominant trait so the ecological pattern reads as blocks.
        # Trait order follows trait_colors; rank_Landmarks orders rows within a block.
        trait_order = {t: i for i, t in enumerate(trait_colors)}
        df["_trait"]  = [family_trait.get(g) for g in df.index]
        df["_torder"] = df["_trait"].map(trait_order).fillna(len(trait_order))
        df = df.sort_values(["_torder", "rank_Landmarks"])
    else:
        df = df.sort_values("rank_Landmarks")

    fig, ax = plt.subplots(figsize=figsize)
    y_pos = np.arange(len(df))[::-1]        # rank 1 at top

    for y, (name, row) in zip(y_pos, df.iterrows()):
        r_lm  = int(row["rank_Landmarks"])
        r_lat = int(row[RANK_COL])
        shift = r_lm - r_lat

        muted = (min_shift is not None) and (abs(shift) < min_shift)
        color = GREY if muted else _group_color(name, level)
        alpha = 0.3  if muted else 1.0
        lw    = 1.2  if muted else 2.4

        # Connecting line, split into segments by trait composition
        if muted:
            ax.plot([r_lm, r_lat], [y, y], color=GREY, lw=lw, alpha=alpha,
                    zorder=1, solid_capstyle="round")
        else:
            span, x0 = r_lat - r_lm, r_lm
            for frac, seg_col in _segments(name, level):
                x1 = x0 + span * frac
                ax.plot([x0, x1], [y, y], color=seg_col, lw=lw, alpha=alpha,
                        zorder=1, solid_capstyle="butt")
                x0 = x1

        ax.scatter(r_lm,  y, s=46, facecolor="white", edgecolor=color,
                   linewidth=1.8, zorder=3, alpha=alpha)
        ax.scatter(r_lat, y, s=46, color=color, zorder=3, alpha=alpha)

        # Bold where the NSM ranks the group more disparate than landmarks do
        ax.text(-0.6, y, name.upper(), ha="right", va="center", fontsize=12,
                color="0.55" if muted else "0.15",
                fontweight="semibold" if shift > 0 else "normal")

    ax.set_yticks([])
    ax.set_xlabel("DISPARITY RANK (1 = MOST DISPARATE)", fontsize=12)
    ax.set_xlim(-0.5, len(df) + (4.5 if level == "family" else 1))
    ax.set_ylim(-1, len(df))

    # Trait block separators + right-margin band labels
    if level == "family":
        traits_seq = df["_trait"].tolist()
        start = 0
        for i in range(1, len(traits_seq) + 1):
            if i == len(traits_seq) or traits_seq[i] != traits_seq[start]:
                t = traits_seq[start]
                y_hi, y_lo = y_pos[start], y_pos[i - 1]
                if i < len(traits_seq):                      # divider below block
                    ax.axhline(y_lo - 0.5, color="0.88", lw=0.8, zorder=0)
                ax.text(len(df) + 3.2, (y_hi + y_lo) / 2,
                        (t or "—").upper(), rotation=-90,
                        ha="center", va="center", fontsize=12,
                        color=trait_colors.get(t, GREY), fontweight="bold")
                start = i

    ax.spines[["left", "right", "top"]].set_visible(False)
    ax.grid(axis="x", color="0.9", lw=0.6, zorder=0)
    ax.set_axisbelow(True)

    handles = [
        Line2D([], [], marker="o", ls="none", color="0.35", markersize=8,
               label="NSM LATENTS"),
        Line2D([], [], marker="o", ls="none", markerfacecolor="white",
               markeredgecolor="0.35", markeredgewidth=1.6, markersize=8,
               label="SPARSE LANDMARKS"),
    ]
    ax.legend(handles=handles, fontsize=12, ncol=2,
              loc="lower right", bbox_to_anchor=(1.0, 1.01),
              frameon=False, borderpad=0.8)

    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Saved → {out_path}")


# ── Family: striped by trait composition, small shifts muted ─────────────────
dumbbell(
    OUT_DIR / "disparity_dimcheck_family.csv",
    OUT_DIR / "disparity_dumbbell_family.png",
    level="family",
    min_shift=MIN_SHIFT,
    figsize=(9, 11),
)

# ── Life history: few groups, show all ───────────────────────────────────────
dumbbell(
    OUT_DIR / "disparity_dimcheck_trait.csv",
    OUT_DIR / "disparity_dumbbell_trait.png",
    level="trait",
    min_shift=None,
    figsize=(7.5, 3.6),
)